# Task C — LLM for School Questions

**Yandex ML Cup — ML track**

**Goal:** answer school-level questions (in Russian) correctly, briefly, and to the point — under a tight compute/time budget, running fully offline in a Docker container.

**Model:** a small **Qwen3** checkpoint (hidden=1024, 28 layers, ~0.6B params) served with **vLLM**.

---

## The core decision — small model + hybrid routing

A 0.6B model is fast and fits the budget, but it's **unreliable at arithmetic** — small LLMs routinely get `47 * 23` wrong. So the solution routes each question:

```
question
   │
   ├─ looks like pure arithmetic?  ── yes ─▶  exact evaluator (Fraction math)
   │                                          → always correct, ~0 ms
   └─ no ─▶  vLLM (Qwen3, greedy decoding)
```

Deterministic problems get a **deterministic solver**; everything else goes to the LLM. This is the same philosophy as Task A (exact solver first, learned model as fallback).

## Part 1 — the exact arithmetic path

Before calling the model, `try_arithmetic` checks whether the question is really just a computation. It's deliberately **conservative** — it only fires when it's confident, otherwise returns `None` and defers to the LLM.

Guards, in order:
1. Short question (`< 140` chars).
2. Contains a compute keyword (`сколько`, `вычисли`, `посчитай`, …).
3. Contains an operator (`+ - * / :`).
4. A number-and-operator substring can be extracted and parses cleanly.

Crucially it evaluates with `fractions.Fraction` over a **whitelisted AST** — not `eval()` — so `1/3 + 1/6` returns the exact `1/2`, and no arbitrary code can run.

In [ ]:
import ast, re
from fractions import Fraction

def _eval_frac(node):
    """Safely evaluate a math AST over exact fractions. Whitelisted nodes only."""
    if isinstance(node, ast.Expression):
        return _eval_frac(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, int):
        return Fraction(node.value, 1)
    if isinstance(node, ast.Constant) and isinstance(node.value, float):
        return Fraction(str(node.value))
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -_eval_frac(node.operand)
    if isinstance(node, ast.BinOp):
        a, b = _eval_frac(node.left), _eval_frac(node.right)
        if isinstance(node.op, ast.Add):  return a + b
        if isinstance(node.op, ast.Sub):  return a - b
        if isinstance(node.op, ast.Mult): return a * b
        if isinstance(node.op, ast.Div):  return a / b
    raise ValueError('unsupported expression')

def try_arithmetic(question):
    q = str(question).lower().strip()
    if len(q) > 140:
        return None
    if not any(w in q for w in ('сколько', 'вычисли', 'посчитай', 'найди значение', 'реши пример')):
        return None
    if not any(op in q for op in ('+', '-', '*', '/', ':')):
        return None
    candidates = re.findall(r'[0-9][0-9\s\+\-\*/:\.,\(\)]{1,90}[0-9\)]', q)
    if not candidates:
        return None
    expr = max(candidates, key=len).replace(',', '.').replace(':', '/')
    expr = re.sub(r'\s+', '', expr)
    if not re.fullmatch(r'[0-9\+\-\*/\.\(\)]+', expr):
        return None
    try:
        val = _eval_frac(ast.parse(expr, mode='eval'))
    except Exception:
        return None
    if val.denominator == 1:
        return f'Ответ: {val.numerator}.'
    return f'Ответ: {val.numerator}/{val.denominator}.'

for q in ['Вычисли 47 * 23', 'Сколько будет 1/3 + 1/6?', 'Кто написал Войну и мир?']:
    print(f'{q!r:45} -> {try_arithmetic(q)}')

## Part 2 — the LLM path (vLLM)

Everything else goes to the Qwen3 model served by **vLLM** (fast batched inference, PagedAttention). Key choices:

| Setting | Value | Why |
|---|---|---|
| `temperature` | **0.0** | greedy — deterministic, reproducible, best for factual QA |
| `max_tokens` | 192 | answers must be short; caps runtime |
| `max_model_len` | 1024 | school questions are short; smaller KV cache = more throughput |
| `dtype` | bfloat16 | fits the GPU, no meaningful quality loss |
| `gpu_memory_utilization` | 0.88 | pack the KV cache as full as is safe |

A **system prompt** forces the style: *answer correctly, briefly, no extra reasoning.* All non-arithmetic questions are batched into a single `llm.generate` call for speed.

In [ ]:
# Chat-template prompt in Qwen's <|im_start|> format
SYSTEM_PROMPT = (
    'Ты помощник для школьных вопросов. '
    'Отвечай правильно, кратко и по делу. '
    'Если нужно решение — дай короткое понятное решение. '
    'Не пиши лишние рассуждения.'
)

def make_prompt(question):
    q = str(question).strip()
    return (
        '<|im_start|>system\n' + SYSTEM_PROMPT + '\n<|im_end|>\n'
        '<|im_start|>user\n'   + q             + '\n<|im_end|>\n'
        '<|im_start|>assistant\n'
    )

# --- vLLM inference (runs in the Docker container with a GPU) ---
# from vllm import LLM, SamplingParams
# llm = LLM(model='/workspace/weights', dtype='bfloat16',
#           gpu_memory_utilization=0.88, max_model_len=1024, trust_remote_code=True)
# sampling = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=192,
#                           stop=['<|im_end|>', '<|endoftext|>'])
# outputs = llm.generate([make_prompt(q) for q in questions], sampling)
print(make_prompt('Кто написал «Войну и мир»?'))

## Part 3 — routing + cleanup

The dispatcher tries arithmetic first, collects everything else for one batched LLM call, then cleans the raw model text (strips `<think>` blocks, chat-template markers, truncates run-ons).

In [ ]:
def clean_answer(text):
    text = str(text or '')
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = text.replace('<|im_end|>', '').replace('<|endoftext|>', '')
    for marker in ('<|im_start|>user', '<|im_start|>system', '<|im_start|>assistant'):
        if marker in text:
            text = text.split(marker, 1)[0]
    return text.strip()

def answer_all(items, llm=None, sampling=None):
    """Route each question: exact arithmetic if possible, else batched vLLM."""
    answers = [None] * len(items)
    llm_idx, llm_prompts = [], []
    for i, item in enumerate(items):
        q = item.get('question', '')
        det = try_arithmetic(q)
        if det is not None:
            answers[i] = det                      # exact path
        else:
            llm_idx.append(i)
            llm_prompts.append(make_prompt(q))    # defer to model
    if llm_prompts and llm is not None:
        outputs = llm.generate(llm_prompts, sampling)   # one batched call
        for i, out in zip(llm_idx, outputs):
            answers[i] = clean_answer(out.outputs[0].text if out.outputs else '')
    return [a if a is not None else '' for a in answers]

## Why this design

- **Small model, not a big one** — the time/compute budget rewards a fast 0.6B model over a slow large one. The gap on hard cases is closed by routing, not by scale.
- **Deterministic solver for deterministic problems** — arithmetic is where small LLMs fail *and* where an exact evaluator is trivial and 100% correct. Take those points for free.
- **Greedy decoding** — for factual short-answer QA there's no benefit to sampling; `temperature=0` is reproducible and avoids random wrong turns.
- **Safe evaluation** — AST whitelist + `Fraction`, never `eval()`, so the arithmetic path can't run arbitrary code and stays exact (no float rounding).
- **Robust I/O** — the container always writes a valid (even empty) output file so a single bad input never zeroes the whole run.

## Summary

| Component | Role |
|---|---|
| `try_arithmetic` | exact `Fraction`-based evaluator for pure math questions |
| Qwen3 0.6B + vLLM | fast batched LLM for everything else, greedy decoding |
| system prompt | enforces short, correct, no-rambling answers |
| `clean_answer` | strips reasoning traces & chat markers from raw output |

**One line:** a hybrid QA system — route deterministic questions to an exact solver, everything else to a small greedy LLM — which beats "just prompt a bigger model" under a strict latency/compute budget.